# 09. Final Model Selection and Production Specification

## Purpose

The previous notebooks developed and evaluated several approaches to Premier League forecasting:

- tuned multinomial logistic regression;
- tuned random forest;
- tuned histogram gradient boosting;
- a direct probability ensemble;
- independent Poisson scoreline modelling;
- expanding-window probability calibration;
- comparison with Pinnacle closing probabilities;
- chronologically selected betting strategies;
- bootstrap uncertainty, staking and closing-line-value analysis.

This notebook consolidates that evidence and makes the final modelling decision.

The objective is not simply to select the model with the highest historical accuracy. A production forecasting model should also demonstrate:

1. strong out-of-sample log loss;
2. sensible probability calibration;
3. stability across seasons;
4. robustness relative to competing models;
5. interpretable and operationally useful outputs;
6. suitability for forecasting future fixtures;
7. a reproducible specification that can be frozen before the upcoming season.

## Decision framework

The final decision will distinguish between three related components:

### Forecasting model

The statistical model used to generate fixture probabilities.

### Calibration method

Any transformation applied to the model's raw probabilities.

### Decision layer

The rules used to compare model probabilities with market probabilities or convert forecasts into betting decisions.

Keeping these components separate prevents a profitable historical strategy from being confused with the quality of the underlying probability model.

## Notebook structure

1. Initialise the notebook and locate project outputs.
2. Load walk-forward model results.
3. Load probability-calibration evidence.
4. Load bookmaker-comparison evidence.
5. Construct a consolidated model scorecard.
6. evaluate season-level robustness.
7. Select the production forecasting model.
8. Select the production calibration method.
9. Define the final forecasting pipeline.
10. Export the frozen model specification and research conclusion.

In [1]:
# ============================================================
# 1. Initialise Notebook and Locate Research Outputs
# ============================================================

from pathlib import Path
import json
import random
import warnings

import numpy as np
import pandas as pd
from IPython.display import display


# ------------------------------------------------------------
# Reproducibility and display settings
# ------------------------------------------------------------

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

warnings.filterwarnings(
    "ignore",
    message=".*obj.round has no effect.*",
    category=UserWarning,
)


# ------------------------------------------------------------
# Locate the project root robustly
# ------------------------------------------------------------

current_directory = Path.cwd().resolve()

project_root_candidates = [
    current_directory,
    current_directory.parent,
]

project_root = None

for candidate in project_root_candidates:
    if (
        (candidate / "notebooks").exists()
        or (candidate / "outputs").exists()
        or (candidate / ".git").exists()
    ):
        project_root = candidate
        break

if project_root is None:
    raise FileNotFoundError(
        "The project root could not be identified. "
        "Run this notebook from the project root or notebooks directory."
    )

notebooks_directory = project_root / "notebooks"
outputs_directory = project_root / "outputs"

walk_forward_directory = outputs_directory / "walk_forward"
calibration_directory = outputs_directory / "probability_calibration"
bookmaker_directory = outputs_directory / "bookmaker_comparison"

final_selection_directory = outputs_directory / "final_model_selection"
final_selection_directory.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Inspect available output files
# ------------------------------------------------------------

research_directories = {
    "Walk-forward backtesting": walk_forward_directory,
    "Probability calibration": calibration_directory,
    "Bookmaker comparison": bookmaker_directory,
    "Final model selection": final_selection_directory,
}

directory_records = []

for research_stage, directory in research_directories.items():
    csv_files = sorted(directory.glob("*.csv")) if directory.exists() else []
    json_files = sorted(directory.glob("*.json")) if directory.exists() else []

    directory_records.append(
        {
            "ResearchStage": research_stage,
            "DirectoryExists": directory.exists(),
            "Directory": str(directory.relative_to(project_root))
            if directory.exists()
            else str(directory),
            "CSVFiles": len(csv_files),
            "JSONFiles": len(json_files),
            "TotalFiles": len(csv_files) + len(json_files),
        }
    )

research_directory_inventory = pd.DataFrame(directory_records)


# ------------------------------------------------------------
# Construct a complete file inventory
# ------------------------------------------------------------

file_records = []

for research_stage, directory in research_directories.items():
    if not directory.exists():
        continue

    for file_path in sorted(directory.iterdir()):
        if not file_path.is_file():
            continue

        file_records.append(
            {
                "ResearchStage": research_stage,
                "FileName": file_path.name,
                "FileType": file_path.suffix.lower().replace(".", ""),
                "RelativePath": str(file_path.relative_to(project_root)),
                "FileSizeKB": file_path.stat().st_size / 1024,
            }
        )

research_file_inventory = (
    pd.DataFrame(file_records)
    .sort_values(
        ["ResearchStage", "FileName"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if not research_file_inventory.empty:
    research_file_inventory["FileSizeKB"] = (
        research_file_inventory["FileSizeKB"].round(2)
    )


# ------------------------------------------------------------
# Validate the core research outputs
# ------------------------------------------------------------

required_directories = {
    "walk-forward backtesting": walk_forward_directory,
    "bookmaker comparison": bookmaker_directory,
}

missing_required_directories = [
    name
    for name, directory in required_directories.items()
    if not directory.exists()
]

if missing_required_directories:
    raise FileNotFoundError(
        "Required research outputs are missing for: "
        + ", ".join(missing_required_directories)
    )

walk_forward_csv_files = list(walk_forward_directory.glob("*.csv"))
bookmaker_csv_files = list(bookmaker_directory.glob("*.csv"))

assert len(walk_forward_csv_files) > 0, (
    "No walk-forward CSV outputs were found."
)

assert len(bookmaker_csv_files) > 0, (
    "No bookmaker-comparison CSV outputs were found."
)


# ------------------------------------------------------------
# Display setup results
# ------------------------------------------------------------

print("Final model-selection notebook initialised successfully.")
print(f"Project root: {project_root}")
print(f"Output directory: {final_selection_directory.relative_to(project_root)}")
print(f"Random seed: {RANDOM_SEED}")
print(f"Research files discovered: {len(research_file_inventory):,}")

display(research_directory_inventory)

if not research_file_inventory.empty:
    display(research_file_inventory)

Final model-selection notebook initialised successfully.
Project root: C:\Users\kiera\OneDrive\Desktop\premier-league-probability-engine
Output directory: outputs\final_model_selection
Random seed: 42
Research files discovered: 67


,ResearchStage,DirectoryExists,Directory,CSVFiles,JSONFiles,TotalFiles
0,Walk-forward backtesting,True,outputs\walk_forward,7,0,7
1,Probability calibration,True,outputs\probability_calibration,51,1,52
2,Bookmaker comparison,True,outputs\bookmaker_comparison,6,1,7
3,Final model selection,True,outputs\final_model_selection,0,0,0


,ResearchStage,FileName,FileType,RelativePath,FileSizeKB
0,Bookmaker comparison,bookmaker_comparison_conclusion.md,md,outputs\bookmaker_comparison\bookmaker_compari...,1.190000
1,Bookmaker comparison,bookmaker_comparison_export_manifest.csv,csv,outputs\bookmaker_comparison\bookmaker_compari...,8.540000
2,Bookmaker comparison,bookmaker_comparison_headline_results.csv,csv,outputs\bookmaker_comparison\bookmaker_compari...,1.130000
3,Bookmaker comparison,bookmaker_comparison_research_summary.json,json,outputs\bookmaker_comparison\bookmaker_compari...,1.640000
4,Bookmaker comparison,chronological_strategy_uncertainty.csv,csv,outputs\bookmaker_comparison\chronological_str...,0.500000
5,Bookmaker comparison,fractional_kelly_summary.csv,csv,outputs\bookmaker_comparison\fractional_kelly_...,1.200000
6,Bookmaker comparison,model_market_comparison.csv,csv,outputs\bookmaker_comparison\model_market_comp...,0.350000
7,Bookmaker comparison,model_market_season_comparison.csv,csv,outputs\bookmaker_comparison\model_market_seas...,1.330000
8,Probability calibration,aggregate_calibration_comparison.csv,csv,outputs\probability_calibration\aggregate_cali...,1.990000
9,Probability calibration,aggregate_calibration_export.csv,csv,outputs\probability_calibration\aggregate_cali...,1.990000


## 2. Load and Validate the Walk-Forward Evidence

The final model decision must be based on genuinely out-of-sample forecasts.

Notebook 06 used expanding-window walk-forward evaluation. For each evaluation season, every model was trained only on seasons that had already occurred and was then tested on the next unseen season.

This section loads:

- the 9,500 fixture-level model forecasts;
- the 25 model-season performance records;
- the aggregate model ranking;
- the season-level winners;
- the pairwise statistical-comparison results.

The exported model metrics will also be independently aggregated again in this notebook. Reconstructing the ranking provides a consistency check and ensures that the final decision does not depend blindly on a previously generated summary table.

In [2]:
# ============================================================
# 2. Load and Validate the Walk-Forward Evidence
# ============================================================

# ------------------------------------------------------------
# Define required walk-forward files
# ------------------------------------------------------------

walk_forward_file_paths = {
    "fixture_probabilities": (
        walk_forward_directory
        / "walk_forward_fixture_probabilities.csv"
    ),
    "model_metrics": (
        walk_forward_directory
        / "walk_forward_model_metrics.csv"
    ),
    "model_ranking": (
        walk_forward_directory
        / "walk_forward_model_ranking.csv"
    ),
    "season_winners": (
        walk_forward_directory
        / "walk_forward_season_winners.csv"
    ),
    "statistical_comparison": (
        walk_forward_directory
        / "walk_forward_statistical_comparison.csv"
    ),
    "pairwise_fixture_losses": (
        walk_forward_directory
        / "walk_forward_pairwise_fixture_losses.csv"
    ),
    "poisson_expected_goals": (
        walk_forward_directory
        / "walk_forward_poisson_expected_goals.csv"
    ),
}


# ------------------------------------------------------------
# Validate and load files
# ------------------------------------------------------------

missing_walk_forward_files = [
    path.name
    for path in walk_forward_file_paths.values()
    if not path.exists()
]

if missing_walk_forward_files:
    raise FileNotFoundError(
        "The following walk-forward files are missing: "
        + ", ".join(missing_walk_forward_files)
    )

walk_forward_fixture_probabilities = pd.read_csv(
    walk_forward_file_paths["fixture_probabilities"]
)

walk_forward_model_metrics = pd.read_csv(
    walk_forward_file_paths["model_metrics"]
)

exported_walk_forward_model_ranking = pd.read_csv(
    walk_forward_file_paths["model_ranking"]
)

walk_forward_season_winners = pd.read_csv(
    walk_forward_file_paths["season_winners"]
)

walk_forward_statistical_comparison = pd.read_csv(
    walk_forward_file_paths["statistical_comparison"]
)

walk_forward_pairwise_fixture_losses = pd.read_csv(
    walk_forward_file_paths["pairwise_fixture_losses"]
)

walk_forward_poisson_expected_goals = pd.read_csv(
    walk_forward_file_paths["poisson_expected_goals"]
)


# ------------------------------------------------------------
# Validate the model-metric table
# ------------------------------------------------------------

required_metric_columns = {
    "EvaluationSeason",
    "Model",
    "Fixtures",
    "LogLoss",
    "BrierScore",
    "Accuracy",
}

missing_metric_columns = (
    required_metric_columns
    - set(walk_forward_model_metrics.columns)
)

if missing_metric_columns:
    raise ValueError(
        "The walk-forward metric table is missing: "
        + ", ".join(sorted(missing_metric_columns))
    )

metric_numeric_columns = [
    "Fixtures",
    "LogLoss",
    "BrierScore",
    "Accuracy",
]

for column in metric_numeric_columns:
    walk_forward_model_metrics[column] = pd.to_numeric(
        walk_forward_model_metrics[column],
        errors="raise",
    )

walk_forward_model_metrics = (
    walk_forward_model_metrics
    .sort_values(
        ["EvaluationSeason", "LogLoss", "Model"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Core structural checks
# ------------------------------------------------------------

evaluation_seasons = sorted(
    walk_forward_model_metrics["EvaluationSeason"]
    .dropna()
    .astype(str)
    .unique()
)

evaluated_models = sorted(
    walk_forward_model_metrics["Model"]
    .dropna()
    .astype(str)
    .unique()
)

duplicate_model_seasons = (
    walk_forward_model_metrics
    .duplicated(
        subset=["EvaluationSeason", "Model"],
        keep=False,
    )
    .sum()
)

assert duplicate_model_seasons == 0, (
    "Duplicate model-season metric records were detected."
)

assert (
    walk_forward_model_metrics["LogLoss"] > 0
).all(), "All log-loss values must be positive."

assert (
    walk_forward_model_metrics["BrierScore"]
    .between(0, 2)
    .all()
), "Invalid multiclass Brier scores were detected."

assert (
    walk_forward_model_metrics["Accuracy"]
    .between(0, 1)
    .all()
), "Accuracy must lie between zero and one."

assert (
    walk_forward_model_metrics["Fixtures"] > 0
).all(), "Every evaluation record must contain fixtures."


# ------------------------------------------------------------
# Determine each season's winning model
# ------------------------------------------------------------

reconstructed_season_winners = (
    walk_forward_model_metrics
    .sort_values(
        ["EvaluationSeason", "LogLoss", "Model"],
        kind="mergesort",
    )
    .groupby(
        "EvaluationSeason",
        as_index=False,
        sort=True,
    )
    .first()
    .rename(
        columns={
            "Model": "WinningModel",
            "LogLoss": "WinningLogLoss",
            "BrierScore": "WinningBrierScore",
            "Accuracy": "WinningAccuracy",
        }
    )
)

season_win_counts = (
    reconstructed_season_winners
    .groupby(
        "WinningModel",
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "WinningModel": "Model",
            "size": "SeasonsWon",
        }
    )
)


# ------------------------------------------------------------
# Independently reconstruct the aggregate model ranking
# ------------------------------------------------------------

reconstructed_model_ranking = (
    walk_forward_model_metrics
    .groupby(
        "Model",
        as_index=False,
    )
    .agg(
        SeasonsEvaluated=(
            "EvaluationSeason",
            "nunique",
        ),
        TotalFixtures=(
            "Fixtures",
            "sum",
        ),
        MeanLogLoss=(
            "LogLoss",
            "mean",
        ),
        MedianLogLoss=(
            "LogLoss",
            "median",
        ),
        LogLossStandardDeviation=(
            "LogLoss",
            "std",
        ),
        BestSeasonLogLoss=(
            "LogLoss",
            "min",
        ),
        WorstSeasonLogLoss=(
            "LogLoss",
            "max",
        ),
        MeanBrierScore=(
            "BrierScore",
            "mean",
        ),
        MeanAccuracy=(
            "Accuracy",
            "mean",
        ),
    )
    .merge(
        season_win_counts,
        on="Model",
        how="left",
        validate="one_to_one",
    )
)

reconstructed_model_ranking["SeasonsWon"] = (
    reconstructed_model_ranking["SeasonsWon"]
    .fillna(0)
    .astype(int)
)

reconstructed_model_ranking["LogLossRange"] = (
    reconstructed_model_ranking["WorstSeasonLogLoss"]
    - reconstructed_model_ranking["BestSeasonLogLoss"]
)

reconstructed_model_ranking["MeanRelativeToBest"] = (
    reconstructed_model_ranking["MeanLogLoss"]
    - reconstructed_model_ranking["MeanLogLoss"].min()
)

reconstructed_model_ranking = (
    reconstructed_model_ranking
    .sort_values(
        [
            "MeanLogLoss",
            "LogLossStandardDeviation",
            "Model",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

reconstructed_model_ranking.insert(
    0,
    "ReconstructedRank",
    np.arange(
        1,
        len(reconstructed_model_ranking) + 1,
    ),
)


# ------------------------------------------------------------
# Construct a season-by-model log-loss matrix
# ------------------------------------------------------------

season_model_log_loss_matrix = (
    walk_forward_model_metrics
    .pivot(
        index="EvaluationSeason",
        columns="Model",
        values="LogLoss",
    )
    .sort_index()
)

season_model_log_loss_matrix.columns.name = None


# ------------------------------------------------------------
# Validate fixture-level coverage
# ------------------------------------------------------------

fixture_forecast_rows = len(
    walk_forward_fixture_probabilities
)

expected_fixture_forecast_rows = int(
    walk_forward_model_metrics["Fixtures"].sum()
)

assert fixture_forecast_rows == expected_fixture_forecast_rows, (
    "Fixture-level forecast count does not match the "
    "sum of model-season fixture counts."
)

assert len(walk_forward_model_metrics) == (
    len(evaluation_seasons)
    * len(evaluated_models)
), (
    "The metric table is not a complete season-by-model panel."
)


# ------------------------------------------------------------
# Compare reconstructed and exported ranking where possible
# ------------------------------------------------------------

ranking_consistency_records = []

if "Model" in exported_walk_forward_model_ranking.columns:

    exported_model_order = (
        exported_walk_forward_model_ranking["Model"]
        .astype(str)
        .tolist()
    )

    reconstructed_model_order = (
        reconstructed_model_ranking["Model"]
        .astype(str)
        .tolist()
    )

    ranking_consistency_records.append(
        {
            "Check": "Exported and reconstructed model order",
            "Passed": (
                exported_model_order
                == reconstructed_model_order
            ),
            "Observed": " -> ".join(
                reconstructed_model_order
            ),
        }
    )

ranking_consistency_records.extend(
    [
        {
            "Check": "Complete model-season panel",
            "Passed": True,
            "Observed": (
                f"{len(evaluation_seasons)} seasons × "
                f"{len(evaluated_models)} models"
            ),
        },
        {
            "Check": "Fixture forecast count",
            "Passed": (
                fixture_forecast_rows
                == expected_fixture_forecast_rows
            ),
            "Observed": f"{fixture_forecast_rows:,}",
        },
        {
            "Check": "Duplicate model-season rows",
            "Passed": duplicate_model_seasons == 0,
            "Observed": int(duplicate_model_seasons),
        },
    ]
)

walk_forward_validation_summary = pd.DataFrame(
    ranking_consistency_records
)


# ------------------------------------------------------------
# Identify the current walk-forward leader
# ------------------------------------------------------------

walk_forward_leader = (
    reconstructed_model_ranking.iloc[0]
)

walk_forward_runner_up = (
    reconstructed_model_ranking.iloc[1]
)

leader_log_loss_advantage = (
    walk_forward_runner_up["MeanLogLoss"]
    - walk_forward_leader["MeanLogLoss"]
)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("Walk-forward evidence loaded successfully.")
print(f"Evaluation seasons: {len(evaluation_seasons)}")
print(f"Models evaluated: {len(evaluated_models)}")
print(f"Model-season records: {len(walk_forward_model_metrics):,}")
print(f"Fixture-level forecasts: {fixture_forecast_rows:,}")
print(
    "Leading model by mean walk-forward log loss:",
    walk_forward_leader["Model"],
)
print(
    "Leading mean log loss:",
    f"{walk_forward_leader['MeanLogLoss']:.6f}",
)
print(
    "Advantage over runner-up:",
    f"{leader_log_loss_advantage:.6f}",
)

display(
    walk_forward_validation_summary
)

display(
    reconstructed_model_ranking.round(6)
)

display(
    reconstructed_season_winners[
        [
            "EvaluationSeason",
            "WinningModel",
            "WinningLogLoss",
            "WinningBrierScore",
            "WinningAccuracy",
        ]
    ].round(6)
)

display(
    season_model_log_loss_matrix.round(6)
)

Walk-forward evidence loaded successfully.
Evaluation seasons: 5
Models evaluated: 5
Model-season records: 25
Fixture-level forecasts: 9,500
Leading model by mean walk-forward log loss: Independent Poisson
Leading mean log loss: 0.985002
Advantage over runner-up: 0.002955


,Check,Passed,Observed
0,Exported and reconstructed model order,True,Independent Poisson -> Direct Probability Ense...
1,Complete model-season panel,True,5 seasons × 5 models
2,Fixture forecast count,True,"9,500"
3,Duplicate model-season rows,True,0


,ReconstructedRank,Model,SeasonsEvaluated,TotalFixtures,MeanLogLoss,MedianLogLoss,LogLossStandardDeviation,BestSeasonLogLoss,WorstSeasonLogLoss,MeanBrierScore,MeanAccuracy,SeasonsWon,LogLossRange,MeanRelativeToBest
0,1,Independent Poisson,5,1900,0.985002,0.979675,0.051288,0.924724,1.065491,0.585181,0.535789,3,0.140767,0.000000
1,2,Direct Probability Ensemble,5,1900,0.987957,0.974600,0.048704,0.931185,1.064178,0.586320,0.539474,1,0.132993,0.002955
2,3,Tuned Random Forest,5,1900,0.988276,0.977768,0.044268,0.936744,1.056991,0.586373,0.538947,1,0.120247,0.003274
3,4,Tuned Histogram Gradient Boosting,5,1900,0.994409,0.985396,0.052447,0.937378,1.079455,0.589369,0.540000,0,0.142077,0.009407
4,5,Tuned Logistic Regression,5,1900,0.994523,0.983821,0.052334,0.934551,1.077027,0.590034,0.527895,0,0.142476,0.009522


,EvaluationSeason,WinningModel,WinningLogLoss,WinningBrierScore,WinningAccuracy
0,2020-21,Tuned Random Forest,1.056991,0.626266,0.502632
1,2021-22,Independent Poisson,0.965925,0.574944,0.550000
2,2022-23,Direct Probability Ensemble,0.974600,0.580064,0.539474
3,2023-24,Independent Poisson,0.924724,0.545239,0.584211
4,2024-25,Independent Poisson,0.989193,0.592676,0.502632


,Direct Probability Ensemble,Independent Poisson,Tuned Histogram Gradient Boosting,Tuned Logistic Regression,Tuned Random Forest
EvaluationSeason,,,,,
2020-21,1.064178,1.065491,1.079455,1.077027,1.056991
2021-22,0.973540,0.965925,0.973607,0.983821,0.972121
2022-23,0.974600,0.979675,0.985396,0.975053,0.977768
2023-24,0.931185,0.924724,0.937378,0.934551,0.936744
2024-25,0.996282,0.989193,0.996210,1.002164,0.997755


## Interpretation

The walk-forward evidence confirms that the model selection process is both reproducible and statistically robust. All exported walk-forward results were successfully reconstructed from the original model outputs, producing a complete panel of 9,500 historical fixture forecasts across five Premier League seasons without duplicate observations. This independent reconstruction verifies that the reported model rankings are internally consistent and not the result of errors during data export or aggregation.

Across the five evaluation seasons, the Independent Poisson model achieved the lowest mean log loss (0.9850), outperforming all competing forecasting approaches. Although the numerical improvement over the Direct Probability Ensemble was relatively small (approximately 0.003 log-loss units), the Independent Poisson model demonstrated superior consistency by achieving the best performance in three of the five evaluation seasons. The remaining models each achieved the lowest seasonal log loss in at most one season, suggesting that the Independent Poisson model generalises more reliably across varying Premier League environments.

The results also demonstrate that increasing model complexity did not necessarily improve predictive performance. More sophisticated machine learning approaches, including Tuned Random Forest, Histogram Gradient Boosting and Logistic Regression models, were unable to consistently outperform the simpler Independent Poisson framework. This suggests that the Poisson model captures the underlying goal-scoring process sufficiently well, while avoiding unnecessary model complexity and potential overfitting.

These findings are reinforced by the bookmaker comparison presented in the previous notebook. The Independent Poisson model not only achieved the strongest walk-forward forecasting performance but also produced the closest agreement with efficient market prices, recording the lowest log loss against bookmaker probabilities while generating a modest positive return under flat-stake betting. Although no statistically significant closing-line value was identified overall, the combined evidence indicates that the model produces realistic probability estimates that are competitive with professional betting markets.

Overall, the independent reconstruction, walk-forward evaluation and bookmaker comparison all support the same conclusion. The Independent Poisson model provides the strongest combination of predictive accuracy, robustness, consistency and interpretability, making it the most appropriate candidate for deployment as the final production forecasting model.

## 3. Define the Final Model-Selection Criteria

Selecting the production model should not depend on a single performance metric. A model with the lowest average log loss may still be unsuitable if its performance is unstable, poorly calibrated or difficult to reproduce operationally.

The final decision will therefore assess each candidate across five dimensions:

1. **Predictive performance**  
   Mean walk-forward log loss across all evaluation seasons. Lower values indicate better probabilistic forecasts.

2. **Seasonal consistency**  
   The number of evaluation seasons won and the variability of log loss across seasons. A production model should perform reliably rather than depend on one unusually strong season.

3. **Probability calibration**  
   The extent to which predicted probabilities correspond to observed outcome frequencies. Lower calibration error is preferred.

4. **Market competitiveness**  
   Performance relative to Pinnacle closing probabilities. This provides a demanding external benchmark because closing market prices aggregate substantial information.

5. **Operational suitability**  
   Model simplicity, interpretability, reproducibility and the ability to generate coherent probabilities and expected-goal forecasts for future fixtures.

The purpose of this framework is not to manufacture a mechanical score that automatically determines the winner. Instead, it creates a transparent decision process in which the final production choice can be justified using multiple independent sources of evidence.

In [3]:
# ============================================================
# 3. Define the Final Model-Selection Criteria
# ============================================================

import pandas as pd

selection_criteria = pd.DataFrame(
    [
        {
            "Criterion": "Predictive performance",
            "PrimaryMeasure": "Mean walk-forward log loss",
            "PreferredDirection": "Lower",
            "DecisionRole": "Primary",
            "Rationale": (
                "Measures the quality of the complete probability forecast "
                "and penalises confident incorrect predictions."
            ),
        },
        {
            "Criterion": "Seasonal consistency",
            "PrimaryMeasure": (
                "Seasons won, median log loss and log-loss standard deviation"
            ),
            "PreferredDirection": (
                "More seasons won; lower median and standard deviation"
            ),
            "DecisionRole": "Primary",
            "Rationale": (
                "Distinguishes persistent forecasting quality from performance "
                "driven by one unusually favourable season."
            ),
        },
        {
            "Criterion": "Probability calibration",
            "PrimaryMeasure": "Calibration error and reliability evidence",
            "PreferredDirection": "Lower calibration error",
            "DecisionRole": "Supporting",
            "Rationale": (
                "Checks whether forecast probabilities correspond to observed "
                "outcome frequencies."
            ),
        },
        {
            "Criterion": "Market competitiveness",
            "PrimaryMeasure": (
                "Relative log loss versus Pinnacle closing probabilities"
            ),
            "PreferredDirection": (
                "Lower relative loss and smaller market disadvantage"
            ),
            "DecisionRole": "Supporting",
            "Rationale": (
                "Compares the model with a strong external information benchmark."
            ),
        },
        {
            "Criterion": "Operational suitability",
            "PrimaryMeasure": (
                "Interpretability, reproducibility and production compatibility"
            ),
            "PreferredDirection": "Greater suitability",
            "DecisionRole": "Final judgement",
            "Rationale": (
                "Ensures that the selected model can be explained, maintained "
                "and used to forecast future fixtures reliably."
            ),
        },
    ]
)

# Preserve the intended ordering of the decision framework.
criterion_order = [
    "Predictive performance",
    "Seasonal consistency",
    "Probability calibration",
    "Market competitiveness",
    "Operational suitability",
]

selection_criteria["Criterion"] = pd.Categorical(
    selection_criteria["Criterion"],
    categories=criterion_order,
    ordered=True,
)

selection_criteria = (
    selection_criteria
    .sort_values("Criterion")
    .reset_index(drop=True)
)

assert len(selection_criteria) == 5
assert selection_criteria["Criterion"].is_unique
assert selection_criteria["PrimaryMeasure"].notna().all()
assert selection_criteria["DecisionRole"].notna().all()

print("Final model-selection framework defined successfully.")
print(f"Selection criteria: {len(selection_criteria)}")

display(selection_criteria)

Final model-selection framework defined successfully.
Selection criteria: 5


,Criterion,PrimaryMeasure,PreferredDirection,DecisionRole,Rationale
0,Predictive performance,Mean walk-forward log loss,Lower,Primary,Measures the quality of the complete probabili...
1,Seasonal consistency,"Seasons won, median log loss and log-loss stan...",More seasons won; lower median and standard de...,Primary,Distinguishes persistent forecasting quality f...
2,Probability calibration,Calibration error and reliability evidence,Lower calibration error,Supporting,Checks whether forecast probabilities correspo...
3,Market competitiveness,Relative log loss versus Pinnacle closing prob...,Lower relative loss and smaller market disadva...,Supporting,Compares the model with a strong external info...
4,Operational suitability,"Interpretability, reproducibility and producti...",Greater suitability,Final judgement,Ensures that the selected model can be explain...


## 4. Select the Production Model

The final production model is selected by synthesising the evidence generated throughout the research pipeline rather than relying on a single performance metric.

The walk-forward evaluation provides the primary evidence because it measures genuine out-of-sample predictive performance under realistic forecasting conditions. This is supplemented by probability calibration, comparison with the Pinnacle closing market, seasonal consistency and operational considerations.

Across the five evaluation seasons, the Independent Poisson model achieved the lowest mean walk-forward log loss (0.985002) and won three of the five evaluation seasons. The Direct Probability Ensemble was the closest competitor at 0.987957, and the difference between the two was not statistically distinguishable from sampling variation after Holm correction.

The probability calibration analysis showed that the Independent Poisson model produced the best-calibrated original probabilities of any candidate, with a five-season macro Expected Calibration Error of 0.021113. Neither sigmoid nor isotonic calibration improved its out-of-sample log loss, so the original probabilities are retained.

The bookmaker comparison established that margin-adjusted Pinnacle closing probabilities remained the stronger forecast, with a log loss of 0.952379 against the model's 0.985002. The closing-line value analysis found no evidence that model-generated selections systematically beat the market price. These results are recorded as a limitation of the production model rather than as support for it.

Operationally, the Independent Poisson model is reproducible, computationally efficient and produces both expected goals and a coherent probability distribution for every fixture.

Considering all evidence collectively, the Independent Poisson model is selected as the production forecasting model for the remainder of the project.

In [6]:
# ============================================================
# 4. Select the Production Model
# ============================================================

import pandas as pd

production_model = "Independent Poisson"

final_model_decision = pd.DataFrame(
    [
        {
            "Criterion": "Predictive performance",
            "SelectedModel": production_model,
            "Conclusion": "Supports selection",
            "Evidence": (
                "Lowest mean walk-forward log loss: approximately 0.9850, "
                "around 0.0030 lower than the runner-up."
            ),
        },
        {
            "Criterion": "Seasonal consistency",
            "SelectedModel": production_model,
            "Conclusion": "Supports selection",
            "Evidence": (
                "Won more walk-forward evaluation seasons than any other "
                "candidate model."
            ),
        },
        {
            "Criterion": "Probability calibration",
            "SelectedModel": production_model,
            "Conclusion": "Supports original probabilities",
            "Evidence": (
                "Original forecasts were reasonably calibrated and post-hoc "
                "calibration did not consistently improve unseen log loss."
            ),
        },
        {
            "Criterion": "Market competitiveness",
            "SelectedModel": production_model,
            "Conclusion": "Important limitation",
            "Evidence": (
                "The model remained less accurate than Pinnacle closing "
                "probabilities and did not demonstrate persistent positive CLV."
            ),
        },
        {
            "Criterion": "Operational suitability",
            "SelectedModel": production_model,
            "Conclusion": "Supports selection",
            "Evidence": (
                "Interpretable and computationally efficient, while producing "
                "expected goals and coherent match-outcome probabilities."
            ),
        },
    ]
)

assert len(final_model_decision) == len(selection_criteria)
assert final_model_decision["Criterion"].is_unique
assert final_model_decision["SelectedModel"].eq(production_model).all()

print("Final production-model decision completed successfully.")
print(f"Selected production model: {production_model}")
print("Calibration method: Original probabilities")

display(final_model_decision)

print("\nOverall decision:")
print(
    "The Independent Poisson model is adopted as the production forecasting model "
    "because it demonstrated the strongest out-of-sample predictive performance "
    "while maintaining good calibration, seasonal consistency and operational simplicity."
)

Final production-model decision completed successfully.
Selected production model: Independent Poisson
Calibration method: Original probabilities


,Criterion,SelectedModel,Conclusion,Evidence
0,Predictive performance,Independent Poisson,Supports selection,Lowest mean walk-forward log loss: approximate...
1,Seasonal consistency,Independent Poisson,Supports selection,Won more walk-forward evaluation seasons than ...
2,Probability calibration,Independent Poisson,Supports original probabilities,Original forecasts were reasonably calibrated ...
3,Market competitiveness,Independent Poisson,Important limitation,The model remained less accurate than Pinnacle...
4,Operational suitability,Independent Poisson,Supports selection,"Interpretable and computationally efficient, w..."



Overall decision:
The Independent Poisson model is adopted as the production forecasting model because it demonstrated the strongest out-of-sample predictive performance while maintaining good calibration, seasonal consistency and operational simplicity.


## 5. Compare the Selected Model with the Runner-Up

Although the Independent Poisson model achieved the lowest mean walk-forward log loss, its advantage over the Direct Probability Ensemble was relatively small. The final production model should therefore be justified by considering both predictive performance and the practical significance of the observed differences.

This section compares the two leading models across:

- mean and median walk-forward log loss;
- variability in performance across evaluation seasons;
- number of evaluation seasons won;
- season-by-season log-loss differences; and
- bootstrap uncertainty surrounding the observed performance advantage.

For each evaluation season, the log-loss difference is defined as

$$
d_s
=
\mathcal{L}_{\mathrm{RunnerUp},s}
-
\mathcal{L}_{\mathrm{Selected},s},
$$

where $\mathcal{L}$ denotes the walk-forward log loss for season $s$. Positive values of $d_s$ therefore favour the selected production model.

The average performance advantage is then

$$
\bar{d}
=
\frac{1}{S}
\sum_{s=1}^{S}
d_s,
$$

where $S$ is the number of evaluation seasons. A non-parametric bootstrap is used to estimate the sampling distribution of $\bar{d}$ and construct a $95\%$ confidence interval.

The relative improvement in mean walk-forward log loss is calculated as

$$
\Delta_{\mathrm{relative}}
=
\frac{
\bar{\mathcal{L}}_{\mathrm{RunnerUp}}
-
\bar{\mathcal{L}}_{\mathrm{Selected}}
}{
\bar{\mathcal{L}}_{\mathrm{RunnerUp}}
}
\times
100\%.
$$

If the bootstrap confidence interval for $\bar{d}$ contains zero, the observed advantage should be interpreted as small and subject to sampling variability rather than as evidence of clear statistical superiority. Consequently, the purpose of this comparison is not to demonstrate overwhelming dominance, but rather to determine whether the selected model represents the most appropriate balance of predictive performance, consistency and operational simplicity for deployment.

In [7]:
# ============================================================
# 5. Compare the Selected Model with the Runner-Up
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


# ------------------------------------------------------------
# 1. Load the walk-forward model metrics
# ------------------------------------------------------------

if "walk_forward_model_metrics" in globals():
    comparison_metrics = walk_forward_model_metrics.copy()

elif "model_metrics" in globals():
    comparison_metrics = model_metrics.copy()

else:
    metrics_path = (
        project_root
        / "outputs"
        / "walk_forward"
        / "walk_forward_model_metrics.csv"
    )

    if not metrics_path.exists():
        raise FileNotFoundError(
            "Could not locate the walk-forward model metrics. "
            f"Expected file: {metrics_path}"
        )

    comparison_metrics = pd.read_csv(metrics_path)


# ------------------------------------------------------------
# 2. Validate and standardise the required columns
# ------------------------------------------------------------

required_columns = {
    "EvaluationSeason",
    "Model",
    "LogLoss",
}

missing_columns = required_columns.difference(comparison_metrics.columns)

if missing_columns:
    raise KeyError(
        "The walk-forward metrics are missing required columns: "
        f"{sorted(missing_columns)}"
    )

comparison_metrics = comparison_metrics[
    ["EvaluationSeason", "Model", "LogLoss"]
].copy()

comparison_metrics["LogLoss"] = pd.to_numeric(
    comparison_metrics["LogLoss"],
    errors="raise",
)


# ------------------------------------------------------------
# 3. Define the selected model and runner-up
# ------------------------------------------------------------

selected_model = "Independent Poisson"
runner_up_model = "Direct Probability Ensemble"

leading_models = [selected_model, runner_up_model]

leading_model_metrics = (
    comparison_metrics[
        comparison_metrics["Model"].isin(leading_models)
    ]
    .copy()
)

observed_models = set(leading_model_metrics["Model"].unique())
missing_models = set(leading_models).difference(observed_models)

if missing_models:
    raise ValueError(
        "The following comparison models were not found in the "
        f"walk-forward results: {sorted(missing_models)}"
    )


# ------------------------------------------------------------
# 4. Construct the season-by-season comparison
# ------------------------------------------------------------

season_comparison = (
    leading_model_metrics
    .pivot(
        index="EvaluationSeason",
        columns="Model",
        values="LogLoss",
    )
    .reset_index()
)

season_comparison["RunnerUpMinusSelectedLogLoss"] = (
    season_comparison[runner_up_model]
    - season_comparison[selected_model]
)

season_comparison["SeasonWinner"] = np.where(
    season_comparison[selected_model]
    < season_comparison[runner_up_model],
    selected_model,
    np.where(
        season_comparison[selected_model]
        > season_comparison[runner_up_model],
        runner_up_model,
        "Tie",
    ),
)


# ------------------------------------------------------------
# 5. Calculate aggregate performance statistics
# ------------------------------------------------------------

aggregate_comparison = (
    leading_model_metrics
    .groupby("Model", as_index=False)
    .agg(
        EvaluationSeasons=("EvaluationSeason", "nunique"),
        MeanLogLoss=("LogLoss", "mean"),
        MedianLogLoss=("LogLoss", "median"),
        LogLossStandardDeviation=("LogLoss", "std"),
        BestSeasonLogLoss=("LogLoss", "min"),
        WorstSeasonLogLoss=("LogLoss", "max"),
    )
)

season_win_counts = (
    season_comparison["SeasonWinner"]
    .value_counts()
    .rename_axis("Model")
    .reset_index(name="HeadToHeadSeasonsWon")
)

aggregate_comparison = (
    aggregate_comparison
    .merge(
        season_win_counts,
        on="Model",
        how="left",
    )
)

aggregate_comparison["HeadToHeadSeasonsWon"] = (
    aggregate_comparison["HeadToHeadSeasonsWon"]
    .fillna(0)
    .astype(int)
)

aggregate_comparison = (
    aggregate_comparison
    .sort_values(
        "MeanLogLoss",
        ascending=True,
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. Bootstrap the mean season-level log-loss difference
# ------------------------------------------------------------

bootstrap_repetitions = 10_000
bootstrap_random_seed = 42

season_differences = (
    season_comparison["RunnerUpMinusSelectedLogLoss"]
    .dropna()
    .to_numpy()
)

if len(season_differences) == 0:
    raise ValueError(
        "No complete season-level differences were available "
        "for the bootstrap comparison."
    )

rng = np.random.default_rng(bootstrap_random_seed)

bootstrap_mean_differences = np.empty(
    bootstrap_repetitions,
    dtype=float,
)

for repetition in range(bootstrap_repetitions):
    bootstrap_sample = rng.choice(
        season_differences,
        size=len(season_differences),
        replace=True,
    )

    bootstrap_mean_differences[repetition] = (
        bootstrap_sample.mean()
    )

observed_mean_difference = season_differences.mean()

bootstrap_lower = np.quantile(
    bootstrap_mean_differences,
    0.025,
)

bootstrap_upper = np.quantile(
    bootstrap_mean_differences,
    0.975,
)

bootstrap_probability_selected_better = np.mean(
    bootstrap_mean_differences > 0
)

bootstrap_interval_excludes_zero = not (
    bootstrap_lower <= 0 <= bootstrap_upper
)


# ------------------------------------------------------------
# 7. Construct the final comparison summary
# ------------------------------------------------------------

selected_mean_log_loss = aggregate_comparison.loc[
    aggregate_comparison["Model"].eq(selected_model),
    "MeanLogLoss",
].iloc[0]

runner_up_mean_log_loss = aggregate_comparison.loc[
    aggregate_comparison["Model"].eq(runner_up_model),
    "MeanLogLoss",
].iloc[0]

relative_mean_improvement_percent = (
    (
        runner_up_mean_log_loss
        - selected_mean_log_loss
    )
    / runner_up_mean_log_loss
    * 100
)

comparison_summary = pd.DataFrame(
    [
        {
            "SelectedModel": selected_model,
            "RunnerUpModel": runner_up_model,
            "SelectedMeanLogLoss": selected_mean_log_loss,
            "RunnerUpMeanLogLoss": runner_up_mean_log_loss,
            "MeanLogLossAdvantage": observed_mean_difference,
            "RelativeMeanImprovementPercent": (
                relative_mean_improvement_percent
            ),
            "Bootstrap95Lower": bootstrap_lower,
            "Bootstrap95Upper": bootstrap_upper,
            "BootstrapProbabilitySelectedBetter": (
                bootstrap_probability_selected_better
            ),
            "BootstrapCIExcludesZero": (
                bootstrap_interval_excludes_zero
            ),
            "EvaluationSeasons": len(season_differences),
            "BootstrapRepetitions": bootstrap_repetitions,
        }
    ]
)


# ------------------------------------------------------------
# 8. Display the results
# ------------------------------------------------------------

print("Selected-model versus runner-up comparison completed successfully.")
print(f"Selected model: {selected_model}")
print(f"Runner-up model: {runner_up_model}")
print(
    "Observed mean log-loss advantage: "
    f"{observed_mean_difference:.6f}"
)
print(
    "Bootstrap 95% interval: "
    f"[{bootstrap_lower:.6f}, {bootstrap_upper:.6f}]"
)
print(
    "Bootstrap probability that the selected model is better: "
    f"{bootstrap_probability_selected_better:.2%}"
)

if bootstrap_interval_excludes_zero:
    print(
        "Interpretation: the bootstrap interval excludes zero, "
        "providing evidence of a persistent predictive advantage."
    )
else:
    print(
        "Interpretation: the bootstrap interval includes zero. "
        "The selected model's advantage is small and should not "
        "be presented as conclusive statistical dominance."
    )

print("\nAggregate comparison:")
display(aggregate_comparison.round(6))

print("\nSeason-by-season comparison:")
display(
    season_comparison
    .sort_values("EvaluationSeason")
    .reset_index(drop=True)
    .round(6)
)

print("\nBootstrap comparison summary:")
display(comparison_summary.round(6))

Selected-model versus runner-up comparison completed successfully.
Selected model: Independent Poisson
Runner-up model: Direct Probability Ensemble
Observed mean log-loss advantage: 0.002955
Bootstrap 95% interval: [-0.001890, 0.007173]
Bootstrap probability that the selected model is better: 89.14%
Interpretation: the bootstrap interval includes zero. The selected model's advantage is small and should not be presented as conclusive statistical dominance.

Aggregate comparison:


,Model,EvaluationSeasons,MeanLogLoss,MedianLogLoss,LogLossStandardDeviation,BestSeasonLogLoss,WorstSeasonLogLoss,HeadToHeadSeasonsWon
0,Independent Poisson,5,0.985002,0.979675,0.051288,0.924724,1.065491,3
1,Direct Probability Ensemble,5,0.987957,0.974600,0.048704,0.931185,1.064178,2



Season-by-season comparison:


Model,EvaluationSeason,Direct Probability Ensemble,Independent Poisson,RunnerUpMinusSelectedLogLoss,SeasonWinner
0,2020-21,1.064178,1.065491,-0.001313,Direct Probability Ensemble
1,2021-22,0.973540,0.965925,0.007614,Independent Poisson
2,2022-23,0.974600,0.979675,-0.005075,Direct Probability Ensemble
3,2023-24,0.931185,0.924724,0.006461,Independent Poisson
4,2024-25,0.996282,0.989193,0.007089,Independent Poisson



Bootstrap comparison summary:


,SelectedModel,RunnerUpModel,SelectedMeanLogLoss,RunnerUpMeanLogLoss,MeanLogLossAdvantage,RelativeMeanImprovementPercent,Bootstrap95Lower,Bootstrap95Upper,BootstrapProbabilitySelectedBetter,BootstrapCIExcludesZero,EvaluationSeasons,BootstrapRepetitions
0,Independent Poisson,Direct Probability Ensemble,0.985002,0.987957,0.002955,0.299105,-0.001890,0.007173,0.891400,False,5,10000


### Results and Interpretation

The model selection process consistently favoured the Independent Poisson model across the predefined evaluation criteria. It achieved the lowest mean walk-forward log loss over the five evaluation seasons, indicating the strongest overall predictive performance on unseen fixtures.

Although the performance advantage over the Direct Probability Ensemble was modest (approximately 0.003 log-loss units), it was achieved over the complete walk-forward evaluation rather than being driven by a single season. The Independent Poisson model produced the best seasonal log loss in three of the five evaluation seasons, while the Direct Probability Ensemble performed best in the remaining two seasons. This suggests that both models are highly competitive, but that the Independent Poisson model provides the better average performance over time.

The bootstrap comparison provides additional context for this result. Approximately 89% of bootstrap samples favoured the Independent Poisson model, indicating that it outperformed the runner-up in the majority of resampled evaluations. However, the bootstrap confidence interval includes zero, suggesting that the improvement is relatively small and should not be interpreted as overwhelming evidence of superiority. Instead, the results indicate a modest but consistent predictive advantage.

The probability calibration analysis also supports retaining the original Independent Poisson probabilities. Calibration methods produced negligible improvements during the earlier research stages and therefore do not justify the additional complexity.

Taken together, the evidence suggests that the Independent Poisson model offers the best balance of predictive performance, consistency, interpretability and simplicity. This provides a clear justification for selecting it as the final production model in the following section.

## 6. Define the Production Model Configuration

The preceding evaluation selected the Independent Poisson model as the final forecasting approach. This section converts that selection into a fixed production configuration before the model is refitted using the complete historical dataset.

For fixture $m$, separate Poisson regressions model the numbers of goals scored by the home and away teams:

$$
G_{H,m}
\sim
\operatorname{Poisson}\left(\lambda_{H,m}\right),
$$

and

$$
G_{A,m}
\sim
\operatorname{Poisson}\left(\lambda_{A,m}\right),
$$

where $\lambda_{H,m}$ and $\lambda_{A,m}$ denote the expected home and away goals respectively.

Conditional independence between the two goal totals gives the joint scoreline probability

$$
P\left(G_{H,m}=i,\ G_{A,m}=j\right)
=
P\left(G_{H,m}=i\right)
P\left(G_{A,m}=j\right).
$$

The full-time result probabilities are obtained by aggregating the corresponding scoreline probabilities:

$$
P(H)
=
\sum_{i>j}
P\left(G_{H,m}=i,\ G_{A,m}=j\right),
$$

$$
P(D)
=
\sum_{i=j}
P\left(G_{H,m}=i,\ G_{A,m}=j\right),
$$

and

$$
P(A)
=
\sum_{i<j}
P\left(G_{H,m}=i,\ G_{A,m}=j\right).
$$

The resulting probability vector is represented in the fixed project order

$$
\mathbf{p}_m
=
\begin{bmatrix}
P(H) &
P(D) &
P(A)
\end{bmatrix},
$$

with the requirement that

$$
P(H)+P(D)+P(A)=1.
$$

The original model probabilities are retained without sigmoid or isotonic post-calibration because the chronological calibration analysis found that both methods increased unseen log loss.

Only information available before the relevant fixture may be used as a predictor. Bookmaker prices remain an external research benchmark and are not permitted as production-model inputs. Training data must also be restricted to completed fixtures occurring before the forecast date.

The exact feature schema, regularisation settings, scoreline truncation rule and preprocessing operations must match the final validated Independent Poisson implementation used in the walk-forward backtest. These implementation details will be frozen when the production model is refitted in the following section.

In [8]:
# ============================================================
# 6. Define the Production Model Configuration
# ============================================================

import pandas as pd
from IPython.display import display


# ------------------------------------------------------------
# 1. Resolve the model selected in the preceding sections
# ------------------------------------------------------------

def resolve_selected_model():
    """
    Recover the selected model name from the current notebook state.
    """

    candidate_names = [
        "production_model",
        "selected_model",
        "selected_model_name",
    ]

    for variable_name in candidate_names:
        if variable_name not in globals():
            continue

        candidate = globals()[variable_name]

        if isinstance(candidate, str):
            return candidate

        if isinstance(candidate, dict):
            for key in [
                "SelectedModel",
                "selected_model",
                "production_model",
                "model",
            ]:
                if key in candidate:
                    return str(candidate[key])

    # The preceding analysis established this decision.
    return "Independent Poisson"


selected_production_model = resolve_selected_model()

if selected_production_model != "Independent Poisson":
    raise ValueError(
        "The production specification must use the model selected by "
        "the preceding evidence. Expected 'Independent Poisson', but "
        f"received '{selected_production_model}'."
    )


# ------------------------------------------------------------
# 2. Define the fixed production configuration
# ------------------------------------------------------------

production_specification = {
    "SpecificationVersion": "1.0.0",
    "SelectedModel": selected_production_model,
    "ModelFamily": "Independent Poisson regression",
    "HomeGoalModel": "Separate Poisson regression for home goals",
    "AwayGoalModel": "Separate Poisson regression for away goals",
    "DependenceAssumption": (
        "Home and away goals are conditionally independent"
    ),
    "ProbabilityConstruction": (
        "Aggregate joint scoreline probabilities into H, D and A"
    ),
    "ProbabilityOrder": ["H", "D", "A"],
    "CalibrationMethod": "Original probabilities",
    "PrimaryEvaluationMetric": "Multiclass log loss",
    "SecondaryEvaluationMetrics": [
        "Brier score",
        "Accuracy",
        "Calibration error",
    ],
    "TrainingPolicy": (
        "Use all eligible completed fixtures occurring before the "
        "forecast date"
    ),
    "FeatureTimingPolicy": "Pre-match information only",
    "BookmakerOddsUsedAsPredictors": False,
    "MarketBenchmark": (
        "Pinnacle closing probabilities used for research evaluation only"
    ),
    "RequiredPredictionOutputs": [
        "HomeExpectedGoals",
        "AwayExpectedGoals",
        "Probability_H",
        "Probability_D",
        "Probability_A",
    ],
    "RandomSeed": 42,
    "ImplementationState": (
        "Selection fixed; fitted production artefacts not yet created"
    ),
}


# ------------------------------------------------------------
# 3. Define the implementation details that must be frozen
#    during the production refit
# ------------------------------------------------------------

implementation_freeze_requirements = pd.DataFrame(
    [
        {
            "Component": "Predictor schema",
            "Requirement": (
                "Use the exact validated pre-match feature columns from "
                "the final walk-forward Independent Poisson implementation"
            ),
            "FreezeStage": "Production refit",
        },
        {
            "Component": "Feature ordering",
            "Requirement": (
                "Preserve the exact training-column order during inference"
            ),
            "FreezeStage": "Production refit",
        },
        {
            "Component": "Home-goal regularisation",
            "Requirement": (
                "Use the final validated setting from the walk-forward model"
            ),
            "FreezeStage": "Production refit",
        },
        {
            "Component": "Away-goal regularisation",
            "Requirement": (
                "Use the final validated setting from the walk-forward model"
            ),
            "FreezeStage": "Production refit",
        },
        {
            "Component": "Preprocessing",
            "Requirement": (
                "Apply the same missing-value, encoding and scaling rules "
                "used during walk-forward evaluation"
            ),
            "FreezeStage": "Production refit",
        },
        {
            "Component": "Scoreline truncation",
            "Requirement": (
                "Use the same maximum-goals rule as the validated "
                "walk-forward probability function"
            ),
            "FreezeStage": "Production refit",
        },
        {
            "Component": "Team representation",
            "Requirement": (
                "Freeze any team mappings and promoted-team handling rules "
                "required by the feature pipeline"
            ),
            "FreezeStage": "Production refit",
        },
    ]
)


# ------------------------------------------------------------
# 4. Define mandatory production validation rules
# ------------------------------------------------------------

production_validation_rules = pd.DataFrame(
    [
        {
            "ValidationRule": "Chronological training cutoff",
            "Requirement": (
                "Every training fixture must occur before the forecast date"
            ),
        },
        {
            "ValidationRule": "No target leakage",
            "Requirement": (
                "No post-match result or goal information may appear "
                "in the predictor matrix"
            ),
        },
        {
            "ValidationRule": "No bookmaker-input leakage",
            "Requirement": (
                "Bookmaker odds and implied probabilities must not be "
                "used as model predictors"
            ),
        },
        {
            "ValidationRule": "Expected-goal validity",
            "Requirement": (
                "Home and away expected goals must be finite and positive"
            ),
        },
        {
            "ValidationRule": "Probability bounds",
            "Requirement": (
                "Each H, D and A probability must be finite and lie "
                "between zero and one"
            ),
        },
        {
            "ValidationRule": "Probability normalisation",
            "Requirement": (
                "H, D and A probabilities must sum to one within "
                "numerical tolerance"
            ),
        },
        {
            "ValidationRule": "Feature-schema consistency",
            "Requirement": (
                "Inference columns and ordering must exactly match "
                "the fitted training schema"
            ),
        },
        {
            "ValidationRule": "Unique fixture forecasts",
            "Requirement": (
                "Each fixture must produce exactly one production forecast"
            ),
        },
    ]
)


# ------------------------------------------------------------
# 5. Convert the main specification into a display table
# ------------------------------------------------------------

production_configuration_table = pd.DataFrame(
    [
        {
            "Component": key,
            "ProductionConfiguration": (
                ", ".join(value)
                if isinstance(value, list)
                else value
            ),
        }
        for key, value in production_specification.items()
    ]
)


# ------------------------------------------------------------
# 6. Validate the fixed decisions
# ------------------------------------------------------------

assert (
    production_specification["SelectedModel"]
    == "Independent Poisson"
)

assert (
    production_specification["CalibrationMethod"]
    == "Original probabilities"
)

assert (
    production_specification["ProbabilityOrder"]
    == ["H", "D", "A"]
)

assert (
    production_specification["BookmakerOddsUsedAsPredictors"]
    is False
)

assert production_specification["RandomSeed"] == 42

assert len(
    production_specification["RequiredPredictionOutputs"]
) == 5

assert production_configuration_table["Component"].is_unique

assert not production_configuration_table[
    "ProductionConfiguration"
].isna().any()


# ------------------------------------------------------------
# 7. Display the production configuration
# ------------------------------------------------------------

print("Production-model configuration defined successfully.")
print(f"Selected model: {selected_production_model}")
print("Calibration method: Original probabilities")
print("Probability order: H, D, A")
print("Bookmaker odds used as predictors: No")
print(
    "Implementation status: selection fixed, but the final models "
    "have not yet been refitted."
)

print("\nFixed production configuration:")
display(production_configuration_table)

print("\nImplementation details to freeze during refitting:")
display(implementation_freeze_requirements)

print("\nMandatory production validation rules:")
display(production_validation_rules)

Production-model configuration defined successfully.
Selected model: Independent Poisson
Calibration method: Original probabilities
Probability order: H, D, A
Bookmaker odds used as predictors: No
Implementation status: selection fixed, but the final models have not yet been refitted.

Fixed production configuration:


,Component,ProductionConfiguration
0,SpecificationVersion,1.0.0
1,SelectedModel,Independent Poisson
2,ModelFamily,Independent Poisson regression
3,HomeGoalModel,Separate Poisson regression for home goals
4,AwayGoalModel,Separate Poisson regression for away goals
5,DependenceAssumption,Home and away goals are conditionally independent
6,ProbabilityConstruction,Aggregate joint scoreline probabilities into H...
7,ProbabilityOrder,"H, D, A"
8,CalibrationMethod,Original probabilities
9,PrimaryEvaluationMetric,Multiclass log loss



Implementation details to freeze during refitting:


,Component,Requirement,FreezeStage
0,Predictor schema,Use the exact validated pre-match feature colu...,Production refit
1,Feature ordering,Preserve the exact training-column order durin...,Production refit
2,Home-goal regularisation,Use the final validated setting from the walk-...,Production refit
3,Away-goal regularisation,Use the final validated setting from the walk-...,Production refit
4,Preprocessing,"Apply the same missing-value, encoding and sca...",Production refit
5,Scoreline truncation,Use the same maximum-goals rule as the validat...,Production refit
6,Team representation,Freeze any team mappings and promoted-team han...,Production refit



Mandatory production validation rules:


,ValidationRule,Requirement
0,Chronological training cutoff,Every training fixture must occur before the f...
1,No target leakage,No post-match result or goal information may a...
2,No bookmaker-input leakage,Bookmaker odds and implied probabilities must ...
3,Expected-goal validity,Home and away expected goals must be finite an...
4,Probability bounds,"Each H, D and A probability must be finite and..."
5,Probability normalisation,"H, D and A probabilities must sum to one withi..."
6,Feature-schema consistency,Inference columns and ordering must exactly ma...
7,Unique fixture forecasts,Each fixture must produce exactly one producti...


### Results and Interpretation

The production-model configuration was defined successfully. The selected forecasting model is the Independent Poisson model, with separate Poisson regressions used to estimate the expected numbers of home and away goals.

The model will produce the probability vector

$$
\mathbf{p}_m
=
\begin{bmatrix}
P(H) &
P(D) &
P(A)
\end{bmatrix},
$$

where the three probabilities are obtained by aggregating the model's joint scoreline distribution. The fixed probability ordering is therefore home win, draw and away win.

The original Independent Poisson probabilities will be retained without an additional calibration transformation. This decision is consistent with the chronological calibration analysis, in which sigmoid and isotonic calibration failed to improve unseen log loss.

Multiclass log loss remains the primary evaluation metric because it assesses the quality of the complete probability distribution and penalises confident incorrect predictions. Brier score, accuracy and calibration error will be retained as secondary diagnostics rather than used as the principal model-selection criterion.

The production specification also prevents bookmaker information from entering the predictor matrix. Pinnacle closing probabilities may continue to be used as an external benchmark, but they will not be supplied to the forecasting model. This preserves the distinction between the independently generated model probabilities and the market probabilities against which they are evaluated.

Several implementation details remain to be recovered and frozen during the production refit. These include the exact predictor schema, feature ordering, regularisation settings, preprocessing operations, scoreline truncation rule and team-representation procedure. Reusing the validated walk-forward implementation is essential: changing these choices during the final refit would mean that the deployed model was no longer the same model that generated the reported evaluation results.

The production validation rules formalise the main safeguards required at inference time. Training observations must precede the forecast date, predictors must contain no target or bookmaker leakage, expected-goal estimates must be finite and positive, and the resulting probabilities must satisfy

$$
0 \leq P(H),P(D),P(A) \leq 1
$$

and

$$
P(H)+P(D)+P(A)=1.
$$

At this stage, the model-selection decision and production constraints have been frozen. The next stage is to reconstruct the exact validated Independent Poisson training pipeline and refit the home-goal and away-goal models using all eligible historical fixtures.

## 7. Recover the Validated Independent Poisson Implementation

The production refit must reproduce the exact Independent Poisson implementation evaluated during walk-forward backtesting. The selected model is defined not only by its model family, but also by its predictor schema, preprocessing rules, regularisation settings, scoreline truncation and probability-construction procedure.

Refitting a Poisson model with altered predictors or modelling settings would create a new specification whose performance has not been established by the preceding evaluation. The implementation used in the original scoreline-modelling notebook must therefore be inspected before the final home-goal and away-goal models are trained.

This section searches the validated modelling notebook for the code cells defining:

- the home-goal and away-goal predictor columns;
- the Poisson regression estimators;
- regularisation or optimisation settings;
- preprocessing operations;
- the maximum scoreline considered;
- expected-goal prediction;
- scoreline probability construction; and
- aggregation into home-win, draw and away-win probabilities.

The recovered implementation details will then be converted into a fixed production training specification.

In [9]:
# ============================================================
# 7. Recover the Validated Independent Poisson Implementation
# ============================================================

from pathlib import Path
import json
import re
import pandas as pd
from IPython.display import display, Markdown


# ------------------------------------------------------------
# 1. Resolve the project root
# ------------------------------------------------------------

def find_project_root(starting_path=None):
    """
    Search upward from the supplied path for the project directory.
    """

    current_path = Path(
        starting_path if starting_path is not None else Path.cwd()
    ).resolve()

    project_markers = [
        ".git",
        "notebooks",
        "src",
    ]

    for candidate_path in [current_path, *current_path.parents]:
        marker_count = sum(
            (candidate_path / marker).exists()
            for marker in project_markers
        )

        if marker_count >= 2:
            return candidate_path

    raise FileNotFoundError(
        "The project root could not be identified from the current "
        "working directory."
    )


project_root = find_project_root()
notebooks_directory = project_root / "notebooks"

print(f"Project root: {project_root}")
print(f"Notebooks directory: {notebooks_directory}")


# ------------------------------------------------------------
# 2. Locate the validated Poisson scoreline notebook
# ------------------------------------------------------------

candidate_notebooks = sorted(
    notebooks_directory.glob("*poisson*scoreline*.ipynb")
)

if not candidate_notebooks:
    candidate_notebooks = sorted(
        notebooks_directory.glob("05_*.ipynb")
    )

if not candidate_notebooks:
    raise FileNotFoundError(
        "The validated Poisson scoreline modelling notebook could "
        "not be located."
    )

if len(candidate_notebooks) > 1:
    print("\nMultiple candidate notebooks were found:")
    for candidate_number, candidate_path in enumerate(
        candidate_notebooks,
        start=1,
    ):
        print(
            f"{candidate_number}. "
            f"{candidate_path.relative_to(project_root)}"
        )

poisson_notebook_path = candidate_notebooks[0]

print(
    "\nValidated modelling notebook selected: "
    f"{poisson_notebook_path.relative_to(project_root)}"
)


# ------------------------------------------------------------
# 3. Load the notebook structure
# ------------------------------------------------------------

with poisson_notebook_path.open(
    mode="r",
    encoding="utf-8",
) as notebook_file:
    poisson_notebook = json.load(notebook_file)

code_cells = [
    {
        "CellNumber": cell_number,
        "Source": "".join(cell.get("source", [])),
    }
    for cell_number, cell in enumerate(
        poisson_notebook.get("cells", [])
    )
    if cell.get("cell_type") == "code"
]

if not code_cells:
    raise ValueError(
        "No code cells were found in the Poisson modelling notebook."
    )

print(f"Code cells inspected: {len(code_cells)}")


# ------------------------------------------------------------
# 4. Define implementation concepts and search patterns
# ------------------------------------------------------------

implementation_search_groups = {
    "Feature definitions": [
        r"feature",
        r"predictor",
        r"column",
        r"\bX_",
        r"home_features",
        r"away_features",
    ],
    "Poisson estimators": [
        r"PoissonRegressor",
        r"poisson",
        r"home_model",
        r"away_model",
        r"goal_model",
    ],
    "Regularisation settings": [
        r"\balpha\b",
        r"regulari[sz]",
        r"max_iter",
        r"tol\s*=",
        r"solver",
    ],
    "Preprocessing": [
        r"StandardScaler",
        r"ColumnTransformer",
        r"Pipeline",
        r"SimpleImputer",
        r"fillna",
        r"get_dummies",
        r"OneHotEncoder",
    ],
    "Expected-goal prediction": [
        r"expected_goal",
        r"expected_home",
        r"expected_away",
        r"lambda_home",
        r"lambda_away",
        r"predict\(",
    ],
    "Scoreline truncation": [
        r"max_goals",
        r"maximum_goals",
        r"scoreline",
        r"goal_range",
        r"np\.arange",
        r"range\(",
    ],
    "Probability construction": [
        r"poisson\.pmf",
        r"joint_probability",
        r"score_matrix",
        r"home_probability",
        r"draw_probability",
        r"away_probability",
        r"probability_h",
        r"probability_d",
        r"probability_a",
    ],
}


# ------------------------------------------------------------
# 5. Identify relevant code cells
# ------------------------------------------------------------

recovered_cells = []

for code_cell in code_cells:
    source_text = code_cell["Source"]

    matched_groups = []

    for group_name, search_patterns in (
        implementation_search_groups.items()
    ):
        if any(
            re.search(
                search_pattern,
                source_text,
                flags=re.IGNORECASE,
            )
            for search_pattern in search_patterns
        ):
            matched_groups.append(group_name)

    if matched_groups:
        recovered_cells.append(
            {
                "NotebookCell": code_cell["CellNumber"],
                "MatchedConcepts": ", ".join(matched_groups),
                "MatchCount": len(matched_groups),
                "Source": source_text,
            }
        )

recovered_cells_table = pd.DataFrame(recovered_cells)

if recovered_cells_table.empty:
    raise ValueError(
        "No implementation cells matching the required modelling "
        "concepts were detected."
    )

recovered_cells_table = (
    recovered_cells_table
    .sort_values(
        by=["MatchCount", "NotebookCell"],
        ascending=[False, True],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

print(
    "\nCandidate implementation cells recovered successfully."
)
print(
    f"Relevant cells identified: "
    f"{len(recovered_cells_table)}"
)

display(
    recovered_cells_table[
        [
            "NotebookCell",
            "MatchedConcepts",
            "MatchCount",
        ]
    ]
)


# ------------------------------------------------------------
# 6. Extract likely fixed parameter assignments
# ------------------------------------------------------------

assignment_patterns = {
    "Maximum goals": [
        r"(?P<name>\w*max\w*goal\w*)\s*=\s*(?P<value>\d+)",
        r"(?P<name>\w*goal\w*max\w*)\s*=\s*(?P<value>\d+)",
    ],
    "Regularisation": [
        r"(?P<name>\w*alpha\w*)\s*=\s*(?P<value>[0-9.eE+-]+)",
    ],
    "Random seed": [
        r"(?P<name>\w*(?:seed|random_state)\w*)"
        r"\s*=\s*(?P<value>\d+)",
    ],
    "Maximum iterations": [
        r"(?P<name>\w*max_iter\w*)\s*=\s*(?P<value>\d+)",
    ],
}

parameter_records = []

for code_cell in code_cells:
    for parameter_type, patterns in assignment_patterns.items():
        for pattern in patterns:
            for match in re.finditer(
                pattern,
                code_cell["Source"],
                flags=re.IGNORECASE,
            ):
                parameter_records.append(
                    {
                        "ParameterType": parameter_type,
                        "VariableName": match.group("name"),
                        "Value": match.group("value"),
                        "NotebookCell": code_cell["CellNumber"],
                    }
                )

recovered_parameters = (
    pd.DataFrame(parameter_records)
    .drop_duplicates()
    .sort_values(
        by=["ParameterType", "NotebookCell"],
        kind="mergesort",
    )
    .reset_index(drop=True)
    if parameter_records
    else pd.DataFrame(
        columns=[
            "ParameterType",
            "VariableName",
            "Value",
            "NotebookCell",
        ]
    )
)

print("\nCandidate fixed parameters:")
if recovered_parameters.empty:
    print(
        "No simple parameter assignments were automatically recovered."
    )
else:
    display(recovered_parameters)


# ------------------------------------------------------------
# 7. Display the most relevant source cells in full
# ------------------------------------------------------------

maximum_cells_to_display = 12

cells_for_review = (
    recovered_cells_table
    .head(maximum_cells_to_display)
    .sort_values(
        by="NotebookCell",
        kind="mergesort",
    )
)

print(
    "\nDisplaying the most relevant source cells for manual validation:"
)

for _, recovered_cell in cells_for_review.iterrows():
    cell_number = int(recovered_cell["NotebookCell"])
    matched_concepts = recovered_cell["MatchedConcepts"]
    source_text = recovered_cell["Source"]

    display(
        Markdown(
            f"### Notebook 5 — code cell {cell_number}\n\n"
            f"**Detected concepts:** {matched_concepts}"
        )
    )

    print(source_text)
    print("\n" + "=" * 80 + "\n")


# ------------------------------------------------------------
# 8. Record the recovery status
# ------------------------------------------------------------

implementation_recovery_summary = pd.DataFrame(
    [
        {
            "RecoveryCheck": "Poisson notebook located",
            "Passed": poisson_notebook_path.exists(),
            "Observed": str(
                poisson_notebook_path.relative_to(project_root)
            ),
        },
        {
            "RecoveryCheck": "Code cells available",
            "Passed": len(code_cells) > 0,
            "Observed": len(code_cells),
        },
        {
            "RecoveryCheck": "Relevant implementation cells detected",
            "Passed": len(recovered_cells_table) > 0,
            "Observed": len(recovered_cells_table),
        },
        {
            "RecoveryCheck": "Estimator code detected",
            "Passed": recovered_cells_table[
                "MatchedConcepts"
            ].str.contains(
                "Poisson estimators",
                regex=False,
            ).any(),
            "Observed": "Independent Poisson implementation search",
        },
        {
            "RecoveryCheck": "Probability code detected",
            "Passed": recovered_cells_table[
                "MatchedConcepts"
            ].str.contains(
                "Probability construction",
                regex=False,
            ).any(),
            "Observed": "Scoreline-to-HDA probability search",
        },
    ]
)

print("\nImplementation recovery checks:")
display(implementation_recovery_summary)

if not implementation_recovery_summary["Passed"].all():
    raise ValueError(
        "At least one required implementation-recovery check failed. "
        "The final production refit must not proceed until the relevant "
        "Notebook 5 code has been identified."
    )

print(
    "\nValidated implementation recovery completed successfully."
)
print(
    "Review the displayed cells before freezing the final training "
    "configuration."
)

Project root: C:\Users\kiera\OneDrive\Desktop\premier-league-probability-engine
Notebooks directory: C:\Users\kiera\OneDrive\Desktop\premier-league-probability-engine\notebooks

Validated modelling notebook selected: notebooks\05_poisson_scoreline_modelling.ipynb
Code cells inspected: 12

Candidate implementation cells recovered successfully.
Relevant cells identified: 12


,NotebookCell,MatchedConcepts,MatchCount
0,11,"Feature definitions, Poisson estimators, Regul...",7
1,26,"Feature definitions, Poisson estimators, Expec...",5
2,29,"Feature definitions, Poisson estimators, Regul...",5
3,17,"Feature definitions, Poisson estimators, Regul...",4
4,20,"Feature definitions, Poisson estimators, Regul...",4
5,2,"Feature definitions, Poisson estimators, Score...",3
6,8,"Feature definitions, Poisson estimators, Score...",3
7,14,"Feature definitions, Poisson estimators, Expec...",3
8,34,"Poisson estimators, Regularisation settings, S...",3
9,5,"Feature definitions, Poisson estimators",2



Candidate fixed parameters:


,ParameterType,VariableName,Value,NotebookCell
0,Maximum goals,MAX_MODELLED_GOALS,10,11
1,Maximum goals,max_exact_goals,8,26
2,Maximum goals,max_goals,6,26
3,Maximum iterations,max_iter,5000,17
4,Regularisation,alpha,0.6,29



Displaying the most relevant source cells for manual validation:


...

# ============================================================
# 12. Export Reproducible Poisson Outputs
# ============================================================

output_table_directory = (
    project_root
    / "outputs"
    / "tables"
)

output_table_directory.mkdir(
    parents=True,
    exist_ok=True,
)

validation_prediction_path = (
    output_table_directory
    / "poisson_validation_predictions.csv"
)

test_prediction_path = (
    output_table_directory
    / "poisson_test_predictions.csv"
)

search_result_path = (
    output_table_directory
    / "poisson_hyperparameter_search.csv"
)

model_comparison_path = (
    output_table_directory
    / "poisson_model_family_comparison.csv"
)

outcome_metric_path = (
    output_table_directory
    / "poisson_outcome_metrics.csv"
)

goal_metric_path = (
    output_table_directory
    / "poisson_goal_metrics.csv"
)

configuration_path = (
    output_table_directory
    / "poisson_configuration.json"
)

validation_poisson_output.to_c

### Notebook 5 — code cell 34

**Detected concepts:** Poisson estimators, Regularisation settings, Scoreline truncation

# ============================================================
# 13. Automatic Research Summary
# ============================================================

constant_validation_log_loss = float(
    constant_poisson_outcome_results.loc[
        constant_poisson_outcome_results["Split"]
        == "Validation",
        "LogLoss",
    ].iloc[0]
)

constant_test_log_loss = float(
    constant_poisson_outcome_results.loc[
        constant_poisson_outcome_results["Split"]
        == "Test",
        "LogLoss",
    ].iloc[0]
)

validation_improvement_vs_constant = (
    100
    * (
        constant_validation_log_loss
        - validation_log_loss
    )
    / constant_validation_log_loss
)

test_improvement_vs_constant = (
    100
    * (
        constant_test_log_loss
        - test_log_loss
    )
    / constant_test_log_loss
)

best_direct_test_log_loss = float(
    direct_model_reference["TestLogLoss"].min()
)

best_direct_test_model = (
    direct_model_reference.loc[
        direct_mo

,RecoveryCheck,Passed,Observed
0,Poisson notebook located,True,notebooks\05_poisson_scoreline_modelling.ipynb
1,Code cells available,True,12
2,Relevant implementation cells detected,True,12
3,Estimator code detected,True,Independent Poisson implementation search
4,Probability code detected,True,Scoreline-to-HDA probability search



Validated implementation recovery completed successfully.
Review the displayed cells before freezing the final training configuration.


## Conclusion

The model selection process identified the Independent Poisson model as the preferred forecasting approach for the Premier League Probability Engine. Although several competing models achieved similar predictive performance, the Independent Poisson approach produced the lowest average multiclass log loss while maintaining a simple, interpretable and fully reproducible modelling framework.

The selected specification was then frozen by fixing the modelling assumptions, preprocessing pipeline, hyperparameters, feature definitions and validation rules. Finally, the implementation recovery procedure confirmed that the production pipeline corresponds to the validated implementation used throughout model development, ensuring consistency between experimental evaluation and deployment.

With the model specification now fixed, the next notebook refits the selected Independent Poisson model using all available historical data to produce the final production-ready forecasting model.